# 01 — EEG Preprocessing

Loads Helsinki Neonatal EEG (3 EDF files), applies bandpass/notch filtering via MNE, extracts 30-second epochs with 50% overlap, computes per-subject feature vectors (band power, BSR, IBI, SEF95), and saves both tabular features and raw epoch arrays to `datasets/processed/eeg/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn "dvc[gdrive]" pytorch-tabnet \
  xgboost catboost tqdm pyyaml

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!dvc remote add -d gdrive gdrive://19DlpHZ5QCBcFKvfKHprIUhUeVg5bDxU4 --force
!dvc pull
print('Datasets ready.')

In [ ]:
import sys
sys.path.insert(0, '/content/earlyMind')

from src.config import cfg
from src.data.eeg_loader import process_eeg_dataset
import pandas as pd
import numpy as np

# Show config paths
print('EEG raw dir:  ', cfg.paths.eeg_raw)
print('EEG proc dir: ', cfg.paths.eeg_processed)

In [ ]:
# Inspect clinical CSV columns
import pandas as pd
clinical_path = cfg.paths.eeg_raw / 'clinical_information.csv'
if clinical_path.exists():
    df_clin = pd.read_csv(clinical_path)
    print('Clinical CSV columns:', df_clin.columns.tolist())
    display(df_clin.head())
else:
    print('clinical_information.csv not found')

In [ ]:
# Inspect annotations CSV
ann_path = cfg.paths.eeg_raw / 'annotations_2017_A.csv'
if ann_path.exists():
    df_ann = pd.read_csv(ann_path)
    print('Annotations columns:', df_ann.columns.tolist())
    print('Unique subjects with seizures:', df_ann.iloc[:, 0].nunique())
    display(df_ann.head())
else:
    print('annotations_2017_A.csv not found')

In [ ]:
# Run full EEG preprocessing
results = process_eeg_dataset(
    eeg_dir=cfg.paths.eeg_raw,
    output_dir=cfg.paths.eeg_processed,
)

print('\n=== Results ===')
for sid, info in results.items():
    epochs = info['epochs']
    feats  = info['features']
    print(f'  Subject {sid}: epochs={epochs.shape}, features={feats.shape}, '
          f'label={info["label"]}, DQ={info["dq"]:.1f}')

In [ ]:
# Save labels CSV for DataLoader
import pandas as pd
label_rows = []
for sid, info in results.items():
    label_rows.append({'subject_id': sid, 'label': info['label'], 'dq': info['dq']})
label_df = pd.DataFrame(label_rows)
label_df.to_csv(cfg.paths.eeg_processed / 'labels.csv', index=False)
print('Labels saved:')
display(label_df)

In [ ]:
# Quick sanity plot: feature distributions
import matplotlib.pyplot as plt
import numpy as np

feat_names = ['Delta', 'Theta', 'Alpha', 'Beta', 'Total Power',
              'BSR', 'IBI Mean', 'IBI Std', 'SEF95', 'Amp Mean', 'Amp Std']

all_feats = np.stack([info['features'] for info in results.values()])
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
axes = axes.flatten()
for i, name in enumerate(feat_names):
    axes[i].bar([f'S{sid}' for sid in results.keys()], all_feats[:, i])
    axes[i].set_title(name, fontsize=9)
plt.suptitle('EEG Subject Feature Comparison', fontsize=12)
plt.tight_layout()
plt.savefig(cfg.paths.reports / 'eeg_features.png', dpi=100)
plt.show()

In [ ]:
!dvc push
!git add checkpoints/ reports/ datasets/processed/
!git commit -m 'colab: 01_eeg_preprocess completed'
!git push origin main